In [ ]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import BertTokenizerFast
import torch


## 1. Fast Tokenizer và Cơ chế Offset Mapping
Ta sử dụng `BertTokenizerFast` được tối ưu trên Rust để sinh `offset_mapping` cho từng token trong chuỗi.



In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased", local_files_only=True)
question = "What is the capital of France?"
context = "The capital of France is Paris, located on the Seine River."

encoding = tokenizer(
    question,
    context,
    return_offsets_mapping=True,
    return_token_type_ids=True
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]
token_types = encoding["token_type_ids"]

print(f"Total tokens: {len(tokens)}")
for idx, (t, off, tt) in enumerate(zip(tokens[:18], offsets[:18], token_types[:18])):
    print(f"Token {idx:2d}: {t:12s} | Offset: {str(off):10s} | Segment: {tt}")


## 2. Ánh xạ Character Span sang Token Span (Algorithm Core)
Hàm `align_answer_to_tokens` thực hiện ánh xạ character offset của đáp án trong ngữ cảnh sang vị trí `(start_token, end_token)`.
Hàm xác định chính xác vị trí token bắt đầu và kết thúc, đồng thời xử lý các trường hợp biên.



In [ ]:
def align_answer_to_tokens(question, context, answer_start, answer_text, tokenizer, max_length=128):
    encoding = tokenizer(
        question,
        context,
        max_length=max_length,
        truncation="only_second",
        return_offsets_mapping=True,
        return_token_type_ids=True
    )
    offsets = encoding["offset_mapping"]
    seq_ids = encoding.sequence_ids(0)
    answer_end = answer_start + len(answer_text)
    
    start_token = 0
    end_token = 0
    
    context_token_indices = [idx for idx, s_id in enumerate(seq_ids) if s_id == 1]
    if not context_token_indices:
        return 0, 0, encoding
        
    c_start = context_token_indices[0]
    c_end = context_token_indices[-1]
    
    if offsets[c_start][0] > answer_start or offsets[c_end][1] < answer_end:
        return 0, 0, encoding
        
    idx = c_start
    while idx <= c_end and offsets[idx][0] <= answer_start:
        idx += 1
    start_token = idx - 1
    
    idx = c_end
    while idx >= c_start and offsets[idx][1] >= answer_end:
        idx -= 1
    end_token = idx + 1
    
    return start_token, end_token, encoding


## 3. Kiểm thử 4 Trường Hợp Biên (Edge Cases)
Ta tiến hành kiểm tra thuật toán trên các tình huống phức tạp:
- **Case 1**: Ranh giới đáp án nằm chuẩn xác tại các từ nguyên vẹn.
- **Case 2**: Đáp án bắt đầu hoặc kết thúc tại subword token (ví dụ: tiền tố/hậu tố bị WordPiece chia nhỏ).
- **Case 3**: Đáp án nằm ngoài phạm vi truncation (vượt quá `max_length`).
- **Case 4**: Đáp án không có trong đoạn văn (unanswerable / out-of-context).



In [ ]:
test_cases = [
    {
        "desc": "Standard entity answer (Paris)",
        "question": "What is the capital of France?",
        "context": "The capital of France is Paris.",
        "answer_start": 25,
        "answer_text": "Paris"
    },
    {
        "desc": "Subword token answer (photosynthetic)",
        "question": "Which organisms are photosynthetic?",
        "context": "Certain bacteria are photosynthetic organisms producing oxygen.",
        "answer_start": 21,
        "answer_text": "photosynthetic"
    },
    {
        "desc": "Truncated answer beyond max_length limit",
        "question": "Where was the university founded?",
        "context": "The old campus expanded rapidly across Europe over several centuries. " * 8 + "The university was founded in Prague.",
        "answer_start": 595,
        "answer_text": "Prague"
    }
]

for tc in test_cases:
    s_tok, e_tok, enc = align_answer_to_tokens(
        tc["question"], tc["context"], tc["answer_start"], tc["answer_text"], tokenizer, max_length=64
    )
    extracted = tokenizer.decode(enc["input_ids"][s_tok:e_tok+1])
    print(f"Test: {tc['desc']}")
    print(f"  Target: '{tc['answer_text']}' -> Token range: [{s_tok}, {e_tok}]")
    print(f"  Decoded text: '{extracted}'")
    print(f"  Status: {'Truncated to CLS' if (s_tok, e_tok) == (0, 0) else 'Aligned'}")


## 4. Xử lý Truncation với Sliding Window và Stride
Khi văn bản ngữ cảnh dài vượt quá dung lượng xử lý của mô hình, kỹ thuật Sliding Window với `stride` cho phép chia nhỏ context thành nhiều window có vùng giao thoa, đảm bảo không bỏ sót span đáp án tại ranh giới cắt.



In [ ]:
long_context = (
    "Albert Einstein was a German-born theoretical physicist. "
    "He developed the theory of relativity, one of the two pillars of modern physics. "
    "His work is also known for its influence on the philosophy of science. "
    "In 1921, he was awarded the Nobel Prize in Physics for his explanation of the photoelectric effect. "
    "Einstein moved to the United States in 1933 and took a position at the Institute for Advanced Study in Princeton. "
    "He became an American citizen in 1940. He died in Princeton Hospital on April 18, 1955."
)
target_question = "When was Einstein awarded the Nobel Prize?"

sliding_enc = tokenizer(
    target_question,
    long_context,
    max_length=64,
    stride=24,
    truncation="only_second",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    return_tensors="pt"
)

num_windows = sliding_enc["input_ids"].size(0)
print(f"Generated {num_windows} sliding windows for long context.")
for w_idx in range(num_windows):
    win_tokens = tokenizer.convert_ids_to_tokens(sliding_enc["input_ids"][w_idx])
    first_tok = win_tokens[0]
    last_tok = win_tokens[-1]
    print(f"Window {w_idx}: length = {len(win_tokens)} tokens | First: '{first_tok}' | Last: '{last_tok}'")


## 5. Kết luận phần Tokenizer Alignment
1. `BertTokenizerFast` cùng `return_offsets_mapping=True` là công cụ bắt buộc để chuyển đổi không tổn hao giữa character space của dữ liệu gán nhãn và token space của Transformer.
2. Truncation thông thường làm mất triệt để các đáp án xuất hiện ở cuối văn bản dài.
3. Kỹ thuật `stride=128` (hoặc `stride=24` trong bài toán nhỏ) bảo toàn trọn vẹn mọi span đáp án thông qua việc nhân bản câu hỏi ghép với từng phần ngữ cảnh trượt.

